# NextChord AI バックエンド (Google Colab版) 🚀

このノートブックは、NextChordの重いAI処理（音源分離・音声認識・コード解析）をGoogle Colabの強力な無料GPUを使って爆速で処理するための「半クラウド化」用バックエンドです。

### 使い方
1. 上部メニューの **「ランタイム」** > **「ランタイムのタイプを変更」** から、ハードウェアアクセラレータが **T4 GPU** 等になっていることを確認してください。
2. 以下のセルの左上にある **「▶実行」ボタン** をクリックします。
3. 初回は環境構築に約1〜2分かかります。
4. 最後に表示される `https://xxxxx.loca.lt` のようなURLをコピーして、NextChordフロントエンドの「設定（API URL）」に貼り付けてください。

> ⚠️ **注意**: 「Endpoint IP」が表示されたら、その数字をコピーしておいてください。URLを開いた際にパスワードとして入力を求められる場合があります。

In [ ]:
import os
import time
import urllib.request

# 1. GitHubからNextChordの最新コードを取得
if not os.path.exists('nextchord'):
    !git clone https://github.com/crossroad777/nextchord.git
%cd nextchord
!git pull origin main

# 2. システムパッケージとPythonライブラリのインストール
print("\n⏳ 必要なパッケージをインストールしています... (約1分かかります)")
!apt-get update -qq && apt-get install -y ffmpeg -qq
!pip install -q -r requirements.txt

# 3. localtunnel (外部からアクセスするためのトンネル) のインストール
!npm install -g localtunnel

# 4. Endpoint IPの取得 (localtunnelのパスワード代わり)
print("\n" + "="*50)
print("🔑 以下の「Endpoint IP」の数値をコピーしておいてください！")
endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")
print("Endpoint IP:", endpoint_ip)
print("="*50 + "\n")

# 5. FastAPIサーバーをバックグラウンドで起動
get_ipython().system_raw('uvicorn fastapi-backend.main:app --host 0.0.0.0 --port 8000 > server.log 2>&1 &')

# 6. localtunnelをバックグラウンドで起動してURLを取得
get_ipython().system_raw('lt --port 8000 > lt.log 2>&1 &')

# ログにURLが出力されるまで少し待つ
time.sleep(5)

# 7. 接続URLの表示
try:
    with open('lt.log', 'r') as f:
        lt_output = f.read()
        url = lt_output.split("your url is: ")[-1].strip()
        if url:
            print("✅ 準備完了！以下のURLを NextChord の画面の「設定」に貼り付けてください：")
            print(f"\n👉 {url}\n")
        else:
            print("⚠️ URLの取得に時間がかかっています。lt.log を確認してください。")
            print(lt_output)
except Exception as e:
    print("エラーが発生しました:", e)

print("※ このタブを開いたままにしておくと、バックエンドが稼働し続けます。")